# Day 9 — HOL 2: Handling Incremental Data — Bronze & Silver

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 3 — CDF-Based Incremental Loading |
| **Duration** | ~2 hours |
| **Output** | A real CDF-based Bronze→Silver incremental refresh for `payments`, plus a verified check of the watermark-based `orders`/`order_items` pipeline |

This lab has two halves, matching the two strategies this course actually uses:

- **Part 1 (CDF path):** build a real incremental Bronze → Silver refresh for `gbmart.payments`, using Delta CDF + `MERGE` — the general pattern from ILT 3, applied for real (not the SCD2 dimension-history version — that's Day 10's job; this is a simpler "latest values win" upsert).
- **Part 2 (watermark path):** verify the real `orders`/`order_items` Lakeflow Connect pipeline actually picked up a change, using the same checks a production on-call engineer would run.

---

## Part 1 — CDF-Based Incremental Refresh: `payments`

`payments` is a good first CDF target: it's 1:1 with `orders`, and (unlike `customers`/`products`) doesn't need SCD2 history — when a payment record changes, Silver just needs the current value, not every past version. That makes this a plain **upsert** MERGE, the simplest version of the pattern.

> **Why a practice clone, not the real `gbmart.silver.payments`?** This exercise's whole point is watching a MERGE consume a batch of pending changes, then run again and find nothing left. Against the real shared table, only the first person to run this in the whole cohort would ever see a non-zero merge — everyone after (including you, rehearsing before class) would see "0 changed rows" and the exercise would look broken. A `SHALLOW CLONE` in your own schema gives you the same real data and the same real CDF mechanics, but a version history that's entirely yours — repeatable every time, for every student, forever.

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# ─── Personal practice schema — never write directly to shared gbmart tables ────
PRACTICE_SCHEMA = "main.YOUR_SCHEMA"   # ← replace with a schema you own
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

BRONZE_PRACTICE = f"{PRACTICE_SCHEMA}.bronze_payments_practice"
SILVER_PRACTICE = f"{PRACTICE_SCHEMA}.silver_payments_practice"
CONTROL_TABLE   = f"{PRACTICE_SCHEMA}._incremental_control"

# SHALLOW CLONE: same real data as gbmart.bronze/silver.payments right now, but a
# transaction history that's entirely yours — safe to MERGE into repeatedly.
spark.sql(f"CREATE OR REPLACE TABLE {BRONZE_PRACTICE} SHALLOW CLONE gbmart.bronze.payments")
spark.sql(f"CREATE OR REPLACE TABLE {SILVER_PRACTICE} SHALLOW CLONE gbmart.silver.payments")
spark.sql(f"ALTER TABLE {BRONZE_PRACTICE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Practice clones ready: {BRONZE_PRACTICE}, {SILVER_PRACTICE}")

# One control table, one row per Bronze source that's on the CDF path, tracking
# the last Delta *version* of Bronze that Silver has already processed.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        source_table        STRING,
        last_bronze_version  BIGINT,
        last_run_at          TIMESTAMP
    ) USING DELTA
""")

if spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE source_table = 'payments'").count() == 0:
    spark.sql(f"INSERT INTO {CONTROL_TABLE} VALUES ('payments', -1, NULL)")
    print("Control row for 'payments' initialized at version -1 (means: process everything).")
else:
    spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE source_table = 'payments'").show()

In [ ]:
def refresh_silver_payments():
    """
    CDF-based Bronze -> Silver incremental refresh for payments (practice clones).
    Same 6-step shape as every incremental loader in this course:
    read marker -> pull only what changed -> process -> merge -> advance marker.
    """
    # Step 1 — read the control table's last processed Bronze version
    last_version = spark.sql(
        f"SELECT last_bronze_version FROM {CONTROL_TABLE} WHERE source_table = 'payments'"
    ).collect()[0]["last_bronze_version"]
    print(f"Last processed Bronze version: {last_version}")

    # Step 2 — pull only the CDF rows since then
    cdf_df = (
        spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", last_version + 1)
            .table(BRONZE_PRACTICE)
            # Step 3 — drop update_preimage: we only want each row's current state
            .filter("_change_type != 'update_preimage'")
    )
    change_count = cdf_df.count()
    print(f"Changed rows since last run: {change_count}")

    if change_count == 0:
        print("Nothing new — skipping merge and control table update.")
        return

    # Step 4 — same standardize/clean logic Day 5 used for the full load, applied
    # only to this changed subset (adjust column names here if your Day 5 build used
    # slightly different ones — check gbmart.silver.payments's schema first if unsure).
    processed_df = cdf_df \
        .withColumnRenamed("OrderID", "order_id") \
        .withColumnRenamed("PaymentID", "payment_id") \
        .select("payment_id", "order_id") \
        .dropDuplicates(["payment_id"])

    # Step 5 — MERGE into the Silver practice clone (plain upsert, no SCD history needed)
    silver = DeltaTable.forName(spark, SILVER_PRACTICE)
    (silver.alias("tgt")
        .merge(processed_df.alias("src"), "tgt.payment_id = src.payment_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged {processed_df.count()} row(s) into {SILVER_PRACTICE}")

    # Step 6 — advance the control table to the newest Bronze version just processed
    new_version = cdf_df.agg(spark_max("_commit_version")).collect()[0][0]
    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET last_bronze_version = {new_version}, last_run_at = current_timestamp()
        WHERE source_table = 'payments'
    """)
    print(f"Control table advanced to Bronze version {new_version}")

In [ ]:
# Note: if this is the very first run and last_bronze_version = -1, this will
# reprocess ALL of Bronze payments history as "changes" — that's expected and
# correct behavior for a first run; every subsequent run will be truly incremental.
refresh_silver_payments()

In [ ]:
# Run it again immediately with nothing changed — prove it's incremental.
# Expect "Changed rows since last run: 0".
refresh_silver_payments()

## Part 2 — Verify the Watermark-Based Pipeline (`orders` / `order_items`)

Unlike Part 1, you don't build the ingestion yourself here — the real `orders_data_ingestion_cdc` Lakeflow Connect pipeline (Day 2) already owns this. Your job is to **verify** it actually picked up a change, the same way a production on-call engineer would after a source-side update.

> If your instructor makes a change in the source Postgres `globalmart.orders`/`order_items` tables and re-triggers the pipeline during class, run the checks below **after** that happens. If not, run them anyway to see the current state — the counts just won't have moved.

In [ ]:
# Check 1 — current row counts (compare against whatever they were before the
# instructor's source-side change, if one was made).
spark.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM gbmart.bronze.orders
    UNION ALL
    SELECT 'order_items', COUNT(*) FROM gbmart.bronze.order_items
""").display()

In [ ]:
# Check 2 — most recently updated orders. If a change was made, it should be at
# the top, with an updated_at timestamp close to now.
spark.sql("""
    SELECT orderid, customerid, orderchannel, updated_at
    FROM gbmart.bronze.orders
    ORDER BY updated_at DESC
    LIMIT 5
""").display()

In [ ]:
# Check 3 — Delta history. A new version here, timestamped around when the pipeline
# last ran, confirms the pipeline actually wrote something (vs. finding nothing new).
spark.sql("DESCRIBE HISTORY gbmart.bronze.orders") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

## Next Step

Once Bronze `orders`/`order_items` are confirmed current, the next action in a real pipeline is re-running Silver's `orders`/`order_items` incremental notebook so its own `MERGE INTO` picks up whatever just landed — Day 10 covers exactly this for the SCD-tracked tables (`dim_customer`, `dim_product`), and a `fact_sales` incremental refresh.

## Submission Checklist
- [ ] Practice clones `bronze_payments_practice` / `silver_payments_practice` created in your own schema
- [ ] Control table `main.YOUR_SCHEMA._incremental_control` created
- [ ] `refresh_silver_payments()` run once — initial load completed
- [ ] `refresh_silver_payments()` run a second time — confirmed 0 changed rows
- [ ] Part 2 checks run against the real `orders`/`order_items` Bronze tables
- [ ] Notebook run top-to-bottom with no errors